In [3]:
import tensorflow as tf
import pickle as pickle
import numpy as np
import pandas as pd
import glob
from codecarbon import EmissionsTracker
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
tracker = EmissionsTracker()
tracker.start()
print(tf.__version__)

# to include the Repo code without installing in the environment
import sys
sys.path.append('../')

from CBKGE.NN_creation_and_dependencies import *
from CBKGE.NN_preproc import *
from CBKGE.utilities_validation import *
tracker.stop()

[codecarbon INFO @ 16:02:21] [setup] RAM Tracking...
[codecarbon INFO @ 16:02:21] [setup] GPU Tracking...
[codecarbon INFO @ 16:02:21] No GPU found.
[codecarbon INFO @ 16:02:21] [setup] CPU Tracking...
[codecarbon WARNING @ 16:02:21] No CPU tracking mode found. Falling back on CPU constant mode.
[codecarbon INFO @ 16:02:22] CPU Model on constant consumption mode: AMD Ryzen 9 5950X 16-Core Processor
[codecarbon INFO @ 16:02:22] >>> Tracker's metadata:
[codecarbon INFO @ 16:02:22]   Platform system: Linux-6.16.3-76061603-generic-x86_64-with-glibc2.35
[codecarbon INFO @ 16:02:22]   Python version: 3.10.12
[codecarbon INFO @ 16:02:22]   CodeCarbon version: 2.3.5
[codecarbon INFO @ 16:02:22]   Available RAM : 62.686 GB
[codecarbon INFO @ 16:02:22]   CPU count: 32
[codecarbon INFO @ 16:02:22]   CPU model: AMD Ryzen 9 5950X 16-Core Processor
[codecarbon INFO @ 16:02:22]   GPU count: None
[codecarbon INFO @ 16:02:22]   GPU model: None
[codecarbon INFO @ 16:02:25] Energy consumed for RAM : 0.00

2.18.1


7.977479559179778e-09

# Risultati SCoRE GreedSearch

In [4]:
# For each dataset in wiki20m, disrex, nyt10m, nyt10d, wiki20distant load the corresponding pkl file and concatenate the results, adding a column with the dataset name
total_results = []
for dataset in [ "nyt10m", "nyt10d","disrex","wiki20m"]:#"wiki20m",, "wiki20distant","disrex"
    if dataset == "wiki20distant":
        embedding_types = ["average", "boundary", "mean_max"]
        results = []
        for embedding_type in embedding_types:
            with open(f'/home/lucamariotti/Documents/SCoRE/gs_results_results/wiki20distant__results_balanced_gradual_performance.pkl', 'rb') as f:
                arr = pickle.load(f)
                arr = np.array(arr)
                # Add dataset name column
                arr = np.concatenate([arr, np.full((arr.shape[0], 1), dataset)], axis=1)
                results.append(arr)
        results = np.concatenate(results, axis=0)
    else:
        with open(f'/home/lucamariotti/Documents/SCoRE/gs_results/{dataset}__results_balanced_gradual_performance.pkl', 'rb') as f:
            arr = pickle.load(f)
            arr = np.array(arr)
            # Add dataset name column
            arr = np.concatenate([arr, np.full((arr.shape[0], 1), dataset)], axis=1)
            results = arr
    total_results.append(results)
# Concatenate all results
results = np.concatenate(total_results, axis=0)
# Convert to DataFrame
df = pd.DataFrame(results)
df.columns = [
    'micro',
    'Macro',
    'm@100', 'M@100',
    'm@200', 'M@200',
    'm@300', 'M@300',
    'm@1000', 'M@1000',
    'P@R',
    'KNN',
    'Distance',
    'Threshold',
    'CSD Worst',
    'CSD Ideal',
    'KWh',
    'Method',
    'Seed',
    'EmbeddingType',
    'LearningRate',
    'Dataset'
]
# Filter only existing columns: 'micro', 'Macro', 'EmbeddingType', 'Seed'
df_filtered = df[['micro', 'Macro', 'EmbeddingType', 'Seed', 'CSD Worst', 'CSD Ideal', 'KWh', 'Dataset', 'LearningRate', 'P@R', 'KNN', 'Threshold','Method']]
#multiply micro and Macro by 100
df_filtered['micro'] = df_filtered['micro'].astype(float) * 100
df_filtered['Macro'] = df_filtered['Macro'].astype(float) * 100
#Filter only rows with embedding type average 
#df_filtered = df_filtered[df_filtered['EmbeddingType'] == 'average']
# Show the row with max micro and Macro
df_filtered = df_filtered.sort_values(by=['Dataset','Macro', 'micro'], ascending=False)


# Convert columns to float before aggregation
df_filtered['micro'] = df_filtered['micro'].astype(float)
df_filtered['Macro'] = df_filtered['Macro'].astype(float)
df_filtered['CSD Worst'] = df_filtered['CSD Worst'].astype(float).round(2)
df_filtered['CSD Ideal'] = df_filtered['CSD Ideal'].astype(float).round(2)
df_filtered['KWh'] = df_filtered['KWh'].astype(float).round(2)
df_filtered['score']= np.sqrt((1-df_filtered['Macro'].astype(float)/100)*(1-df_filtered['micro'].astype(float)/100)).round(2)
df_filtered['P@R'] = df_filtered['P@R'].astype(float).round(2)


#Now i want to group by value per Embedding type and dataset and compute the mean and std
grouped = df_filtered.groupby([ 'Dataset','EmbeddingType', 'LearningRate','KNN','Threshold','Method']).agg(
    micro_mean=('micro', 'mean'),
    micro_std=('micro', 'std'),
    Macro_mean=('Macro', 'mean'),
    Macro_std=('Macro', 'std'),
    CSD_Worst_mean=('CSD Worst', 'mean'),
    CSD_Worst_std=('CSD Worst', 'std'),
    CSD_Ideal_mean=('CSD Ideal', 'mean'),
    CSD_Ideal_std=('CSD Ideal', 'std'),
    KWh_mean=('KWh', 'mean'),
    KWh_std=('KWh', 'std'),
    score_mean=('score', 'mean'),
    score_std=('score', 'std'),
    PR_mean=('P@R', 'mean'),
    PR_std=('P@R', 'std')
).reset_index()

# Create a table with mean value followed by +- std
grouped['micro_summary'] = grouped.apply(lambda row: f"{row['micro_mean']:.2f} ± {row['micro_std']:.2f}", axis=1)
grouped['Macro_summary'] = grouped.apply(lambda row: f"{row['Macro_mean']:.2f} ± {row['Macro_std']:.2f}", axis=1)
grouped['CSD_Worst_summary'] = grouped.apply(lambda row: f"{row['CSD_Worst_mean']:.2f} ± {row['CSD_Worst_std']:.2f}", axis=1)
grouped['CSD_Ideal_summary'] = grouped.apply(lambda row: f"{row['CSD_Ideal_mean']:.2f} ± {row['CSD_Ideal_std']:.2f}", axis=1)
grouped['KWh_summary'] = grouped.apply(lambda row: f"{row['KWh_mean']:.2f} ± {row['KWh_std']:.2f}", axis=1)
grouped['score_summary'] = grouped.apply(lambda row: f"{row['score_mean']:.2f} ± {row['score_std']:.2f}", axis=1)
grouped['PR_summary'] = grouped.apply(lambda row: f"{row['PR_mean']:.2f} ± {row['PR_std']:.2f}", axis=1)


# Show entire table
# pd.set_option('display.max_rows', None)
# pd.set_option('display.max_columns', None)
# pd.set_option('display.width', None)

summary_table = grouped[['Dataset','EmbeddingType','LearningRate','KNN','Threshold','Method','micro_summary', 'Macro_summary','score_summary', 'CSD_Worst_summary', 'CSD_Ideal_summary', 'KWh_summary', 'PR_summary']]
#select only row where embedding type is 'average'
# summary_table = summary_table.drop(columns=['EmbeddingType'])
#summary_table = grouped[['dataset','EmbeddingType', 'micro_summary', 'Macro_summary', 'CSD_Worst_summary', 'CSD_Ideal_summary', 'KWh_summary']]
#sort by dataset micro_summary and Macro_summary
summary_table = summary_table.sort_values(by=['Dataset','score_summary','PR_summary', 'CSD_Worst_summary'], ascending=[True,True, False, True])

# Print the summary table
summary_table

/tmp/ipykernel_3637181/4126006086.py:50: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtered['micro'] = df_filtered['micro'].astype(float) * 100
/tmp/ipykernel_3637181/4126006086.py:51: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtered['Macro'] = df_filtered['Macro'].astype(float) * 100


,Dataset,EmbeddingType,LearningRate,KNN,Threshold,Method,micro_summary,Macro_summary,score_summary,CSD_Worst_summary,CSD_Ideal_summary,KWh_summary,PR_summary
220,disrex,boundary,0.0005,50,0.3,BCH,75.26 ± 0.72,65.60 ± 0.60,0.29 ± 0.00,0.03 ± 0.00,0.02 ± 0.00,0.13 ± 0.01,0.76 ± 0.01
224,disrex,boundary,0.0005,50,0.4,BCH,75.26 ± 0.72,65.60 ± 0.60,0.29 ± 0.00,0.03 ± 0.00,0.02 ± 0.00,0.14 ± 0.01,0.76 ± 0.01
228,disrex,boundary,0.0005,50,0.5,BCH,75.26 ± 0.72,65.60 ± 0.60,0.29 ± 0.00,0.03 ± 0.00,0.02 ± 0.00,0.14 ± 0.01,0.76 ± 0.01
231,disrex,boundary,0.0005,50,0.5,LH,75.26 ± 0.72,65.60 ± 0.60,0.29 ± 0.00,0.03 ± 0.00,0.02 ± 0.00,0.62 ± 0.05,0.76 ± 0.01
232,disrex,boundary,0.0005,50,0.6,BCH,75.26 ± 0.72,65.60 ± 0.60,0.29 ± 0.00,0.03 ± 0.00,0.02 ± 0.00,0.15 ± 0.01,0.76 ± 0.01
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1077,wiki20m,boundary,0.0005,50,0.7,BH,58.58 ± 6.74,49.28 ± 7.15,0.46 ± 0.07,0.01 ± 0.00,0.01 ± 0.00,1.59 ± 0.57,0.81 ± 0.02
957,wiki20m,average,0.0005,60,0.7,BH,57.76 ± 3.90,47.45 ± 3.81,0.47 ± 0.04,0.01 ± 0.00,0.01 ± 0.00,2.07 ± 0.04,0.82 ± 0.00
1097,wiki20m,boundary,0.0005,60,0.7,BH,57.54 ± 6.85,47.25 ± 7.18,0.47 ± 0.07,0.01 ± 0.00,0.01 ± 0.00,1.74 ± 0.56,0.81 ± 0.02
977,wiki20m,average,0.0005,70,0.7,BH,57.40 ± 4.09,46.63 ± 3.95,0.48 ± 0.04,0.01 ± 0.00,0.01 ± 0.00,2.24 ± 0.05,0.82 ± 0.00


# Hiclre Results

In [5]:
#HICLRE
# Load the pickle file for the HICLRE results for each dataset in wiki20d, disrex, nyt10m, nyt10, wiki20m and concatenate the results, adding a column with the dataset name without adding dataset column
total_results = []
for dataset in ["wiki20m", "disrex", "nyt10m", "nyt10", "wiki20d"]:
    with open(f'/home/lucamariotti/Documents/SCoRE/results_hiclre_101_300/multilabel_bert_{dataset}.pkl', 'rb') as f:
        results = pickle.load(f)
        total_results.extend(results)  # flatten the list of dicts

# Convert to DataFrame
df_hiclre = pd.DataFrame(total_results)
# #Keep test_micro_f1, test_macro_f1, kwh, seed, dataset
df_hiclre = df_hiclre[['test_micro_f1', 'test_macro_f1', 'kwh','dataset']]
# Convert columns to float before aggregation
df_hiclre['test_micro_f1'] = df_hiclre['test_micro_f1'].astype(float)
df_hiclre['test_macro_f1'] = df_hiclre['test_macro_f1'].astype(float)
df_hiclre['kwh'] = df_hiclre['kwh'].astype(float)
#Multiply micro and Macro by 100 
df_hiclre['test_micro_f1'] = df_hiclre['test_micro_f1'] * 100
df_hiclre['test_macro_f1'] = df_hiclre['test_macro_f1'] * 100
#Now i want to group by value per dataset type and compute the mean and std
grouped_hiclre = df_hiclre.groupby('dataset').agg(
    micro_mean=('test_micro_f1', 'mean'),
    micro_std=('test_micro_f1', 'std'),
    Macro_mean=('test_macro_f1', 'mean'),
    Macro_std=('test_macro_f1', 'std'),
    KWh_mean=('kwh', 'mean'),
    KWh_std=('kwh', 'std')
).reset_index()

# Create a table with mean value followed by +- std
grouped_hiclre['micro_summary'] = grouped_hiclre.apply(lambda row: f"{row['micro_mean']:.2f} ± {row['micro_std']:.2f}", axis=1)
grouped_hiclre['Macro_summary'] = grouped_hiclre.apply(lambda row: f"{row['Macro_mean']:.2f} ± {row['Macro_std']:.2f}", axis=1)
grouped_hiclre['KWh_summary'] = grouped_hiclre.apply(lambda row: f"{row['KWh_mean']:.2f} ± {row['KWh_std']:.2f}", axis=1)

# summary_table_hiclre = grouped_hiclre[['dataset', 'micro_summary', 'Macro_summary', 'KWh_summary']]
summary_table_hiclre = grouped_hiclre[['dataset', 'micro_summary', 'Macro_summary']]
print(summary_table_hiclre)


   dataset micro_summary Macro_summary
0   disrex  60.25 ± 0.53  48.96 ± 2.92
1    nyt10  76.38 ± 0.93  22.34 ± 0.57
2   nyt10m  69.10 ± 0.38  36.10 ± 3.97
3  wiki20d  66.77 ± 3.36   6.56 ± 0.53
4  wiki20m  87.41 ± 0.02  86.00 ± 0.07


# Code to measure Wilconxon between Score and other methods

In [6]:
Pare_results= [{'dataset': 'wiki20m',
  'seed': 42,
  'microf1': 0.8349555366643424,
  'macrof1': 0.8367072847145348},
 {'dataset': 'wiki20m',
  'seed': 71,
  'microf1': 0.838344188989058,
  'macrof1': 0.8377673207952135},
 {'dataset': 'wiki20m',
  'seed': 88,
  'microf1': 0.8332989903152689,
  'macrof1': 0.8344744575319456},
 {'dataset': 'wiki20m',
  'seed': 101,
  'microf1': 0.8373061992139464,
  'macrof1': 0.8390195572555864},
 {'dataset': 'wiki20m',
  'seed': 300,
  'microf1': 0.835326061067975,
  'macrof1': 0.8322210080073896},
 {'dataset': 'nyt10d',
  'seed': 42,
  'microf1': 0.8580950710401684,
  'macrof1': 0.4558368219766042},
 {'dataset': 'nyt10d',
  'seed': 71,
  'microf1': 0.8534036171953103,
  'macrof1': 0.42347440009800547},
 {'dataset': 'nyt10d',
  'seed': 88,
  'microf1': 0.8643054688196985,
  'macrof1': 0.4767782062636946},
 {'dataset': 'nyt10d',
  'seed': 101,
  'microf1': 0.8598311642631024,
  'macrof1': 0.47087856851412513},
 {'dataset': 'nyt10d',
  'seed': 300,
  'microf1': 0.8578923563998191,
  'macrof1': 0.46253777131225654},
 {'dataset': 'disrex',
  'seed': 42,
  'microf1': 0.7689816987329892,
  'macrof1': 0.7057807204060438},
 {'dataset': 'disrex',
  'seed': 71,
  'microf1': 0.7845526550124198,
  'macrof1': 0.7109533353358166},
 {'dataset': 'disrex',
  'seed': 88,
  'microf1': 0.7743622523098379,
  'macrof1': 0.7098809897091769},
 {'dataset': 'disrex',
  'seed': 101,
  'microf1': 0.7725149324842541,
  'macrof1': 0.7070240761030276},
 {'dataset': 'disrex',
  'seed': 300,
  'microf1': 0.7612729927007299,
  'macrof1': 0.6969082257352855},
 {'dataset': 'wiki20d',
  'seed': 42,
  'microf1': 0.742630094847475,
  'macrof1': 0.17866516311117045},
 {'dataset': 'wiki20d',
  'seed': 71,
  'microf1': 0.736262063207524,
  'macrof1': 0.17372330274066214},
 {'dataset': 'wiki20d',
  'seed': 88,
  'microf1': 0.7481095983458361,
  'macrof1': 0.16805121245388846},
 {'dataset': 'wiki20d',
  'seed': 101,
  'microf1': 0.7476066790352505,
  'macrof1': 0.18196090354020955},
 {'dataset': 'wiki20d',
  'seed': 300,
  'microf1': 0.7443763583823536,
  'macrof1': 0.17929294490310924},
 {'dataset': 'nyt10m',
  'seed': 42,
  'microf1': 0.7749777268677612,
  'macrof1': 0.4060599660096938},
 {'dataset': 'nyt10m',
  'seed': 71,
  'microf1': 0.7863181671506938,
  'macrof1': 0.43082399619413064},
 {'dataset': 'nyt10m',
  'seed': 88,
  'microf1': 0.7838144459851254,
  'macrof1': 0.43464647280939017},
 {'dataset': 'nyt10m',
  'seed': 101,
  'microf1': 0.7808769945973112,
  'macrof1': 0.4479859845506356},
 {'dataset': 'nyt10m',
  'seed': 300,
  'microf1': 0.7749428553777723,
  'macrof1': 0.437530100319504}]


Hiclre_results=results = [
    {'dataset': 'wiki20m', 'seed': 71, 'microf1': 86.918613, 'macrof1': 85.255784},
    {'dataset': 'wiki20m', 'seed': 88, 'microf1': 87.246451, 'macrof1': 85.520799},
    {'dataset': 'wiki20m', 'seed': 42, 'microf1': 87.598815, 'macrof1': 86.341685},
    {'dataset': 'wiki20m', 'seed': 101, 'microf1': 87.393434, 'macrof1': 85.954852},
    {'dataset': 'wiki20m', 'seed': 300, 'microf1': 87.424996, 'macrof1': 86.054091},

    {'dataset': 'disrex', 'seed': 71, 'microf1': 60.746090, 'macrof1': 51.075662},
    {'dataset': 'disrex', 'seed': 88, 'microf1': 59.918847, 'macrof1': 47.849431},
    {'dataset': 'disrex', 'seed': 42, 'microf1': 61.277827, 'macrof1': 51.244855},
    {'dataset': 'disrex',  'seed': 101, 'microf1': 60.625846, 'macrof1': 51.024049},
    {'dataset': 'disrex',  'seed': 300, 'microf1': 59.870894, 'macrof1': 46.895666},

    {'dataset': 'nyt10m', 'seed': 71, 'microf1': 67.835196, 'macrof1': 33.597089},
    {'dataset': 'nyt10m', 'seed': 88, 'microf1': 68.720605, 'macrof1': 33.700881},
    {'dataset': 'nyt10m', 'seed': 42, 'microf1': 68.697578, 'macrof1': 34.711190},
    {'dataset': 'nyt10m',  'seed': 101, 'microf1': 69.369369, 'macrof1': 38.901892},
    {'dataset': 'nyt10m',  'seed': 300, 'microf1': 68.825301, 'macrof1': 33.293504},

    {'dataset': 'nyt10d', 'seed': 71, 'microf1': 76.121853, 'macrof1': 21.582002},
    {'dataset': 'nyt10d', 'seed': 88, 'microf1': 74.575642, 'macrof1': 18.911261},
    {'dataset': 'nyt10d', 'seed': 42, 'microf1': 75.204991, 'macrof1': 18.396142},
    {'dataset': 'nyt10d',   'seed': 101, 'microf1': 77.034520, 'macrof1': 22.740465},
    {'dataset': 'nyt10d',   'seed': 300, 'microf1': 75.724276, 'macrof1': 21.940828},

    {'dataset': 'wiki20d', 'seed': 71, 'microf1': 67.995262, 'macrof1': 6.757896},
    {'dataset': 'wiki20d', 'seed': 88, 'microf1': 65.159790, 'macrof1': 6.249496},
    {'dataset': 'wiki20d', 'seed': 42, 'microf1': 68.298028, 'macrof1': 6.887623},
    {'dataset': 'wiki20d', 'seed': 101, 'microf1': 64.387468, 'macrof1': 6.183800},
    {'dataset': 'wiki20d', 'seed': 300, 'microf1': 69.145815, 'macrof1': 6.931091}
    
]

Score_results = [
    {'dataset': 'wiki20m', 'seed': 42,  'microf1': 82.755424, 'macrof1': 80.620385},
    {'dataset': 'wiki20m', 'seed': 101, 'microf1': 82.662121, 'macrof1': 80.397290},
    {'dataset': 'wiki20m', 'seed': 300, 'microf1': 82.500881, 'macrof1': 80.008971},
    {'dataset': 'wiki20m', 'seed': 71,  'microf1': 82.278666, 'macrof1': 79.950500},
    {'dataset': 'wiki20m', 'seed': 88,  'microf1': 81.663284, 'macrof1': 79.147313},

    {'dataset': 'nyt10m', 'seed': 42,  'microf1': 76.475850, 'macrof1': 44.311930},
    {'dataset': 'nyt10m', 'seed': 71,  'microf1': 78.064390, 'macrof1': 42.226187},
    {'dataset': 'nyt10m', 'seed': 88,  'microf1': 76.085414, 'macrof1': 41.527295},
    {'dataset': 'nyt10m', 'seed': 101, 'microf1': 76.755370, 'macrof1': 41.304374},
    {'dataset': 'nyt10m', 'seed': 300, 'microf1': 76.560599, 'macrof1': 41.224553},

    {'dataset': 'nyt10d', 'seed': 42,  'microf1': 87.958909, 'macrof1': 49.582342},
    {'dataset': 'nyt10d', 'seed': 101, 'microf1': 87.081259, 'macrof1': 49.255776},
    {'dataset': 'nyt10d', 'seed': 300, 'microf1': 88.621955, 'macrof1': 47.907420},
    {'dataset': 'nyt10d', 'seed': 71,  'microf1': 88.264100, 'macrof1': 47.852272},
    {'dataset': 'nyt10d', 'seed': 88,  'microf1': 86.268657, 'macrof1': 44.267425},

    {'dataset': 'disrex', 'seed': 71,  'microf1': 74.975108, 'macrof1': 66.286088},
    {'dataset': 'disrex', 'seed': 88,  'microf1': 74.533094, 'macrof1': 65.804305},
    {'dataset': 'disrex', 'seed': 300, 'microf1': 74.424866, 'macrof1': 65.627507},
    {'dataset': 'disrex', 'seed': 101, 'microf1': 74.884648, 'macrof1': 65.434527},
    {'dataset': 'disrex', 'seed': 42,  'microf1': 74.413156, 'macrof1': 65.350138}

]



In [7]:
# Show entire table
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

In [8]:
filtered_df = df_filtered[(df_filtered['Dataset'] == "wiki20m") & (df_filtered['Method'] == "LH") & (df_filtered['EmbeddingType'] == "average") & (df_filtered['Threshold'] == "0.4") & (df_filtered['KNN'] == "40")]
filtered_df=filtered_df[['micro','Macro','Seed']]
filtered_df

,micro,Macro,Seed
4321,82.755424,80.620385,42
4741,82.662121,80.397290,101
4881,82.500881,80.008971,300
4461,82.278666,79.950500,71
4601,81.663284,79.147313,88


In [9]:
filtered_df = df_filtered[(df_filtered['Dataset'] == "nyt10m") & (df_filtered['Method'] == "LH") & (df_filtered['EmbeddingType'] == "average") & (df_filtered['Threshold'] == "0.5") & (df_filtered['KNN'] == "70")]
filtered_df=filtered_df[['micro','Macro','Seed']]
filtered_df

,micro,Macro,Seed
137,76.475850,44.311930,42
277,78.064390,42.226187,71
417,76.085414,41.527295,88
557,76.755370,41.304374,101
697,76.560599,41.224553,300


In [10]:
filtered_df = df_filtered[(df_filtered['Dataset'] == "nyt10d") & (df_filtered['Method'] == "LH") & (df_filtered['EmbeddingType'] == "average") & (df_filtered['Threshold'] == "0.3") & (df_filtered['KNN'] == "70")]
filtered_df=filtered_df[['micro','Macro','Seed']]
filtered_df

,micro,Macro,Seed
1535,87.958909,49.582342,42
1955,87.081259,49.255776,101
2095,88.621955,47.907420,300
1675,88.264100,47.852272,71
1815,86.268657,44.267425,88


In [11]:
filtered_df = df_filtered[(df_filtered['Dataset'] == "disrex") & (df_filtered['Method'] == "LH") & (df_filtered['EmbeddingType'] == "average") & (df_filtered['Threshold'] == "0.3") & (df_filtered['KNN'] == "70")]
filtered_df=filtered_df[['micro','Macro','Seed']]
filtered_df

,micro,Macro,Seed
3075,74.975108,66.286088,71
3215,74.533094,65.804305,88
3495,74.424866,65.627507,300
3355,74.884648,65.434527,101
2935,74.413156,65.350138,42


In [12]:
# A parità di dataset e di seed, calcolami il vettore delle differenze della microf1 e della macrof1
# tra il mio metodo Score e Pare, poi fai lo stesso tra Score e Hiclre.
# I risultati sono dentro Score_results, Pare_results e Hiclre_results

def compute_differences(score_results, pare_results, hiclre_results):
    # Convert lists of dicts to DataFrames for a robust merge on dataset+seed
    df_score = pd.DataFrame(score_results)
    df_pare = pd.DataFrame(pare_results)
    df_hiclre = pd.DataFrame(hiclre_results)

    # Normalize Pare (some entries are in [0,1] while others are percentages). Convert to percentages if needed.
    if 'microf1' in df_pare.columns and df_pare['microf1'].max() <= 1.0:
        df_pare['microf1'] = df_pare['microf1'] * 100
    if 'macrof1' in df_pare.columns and df_pare['macrof1'].max() <= 1.0:
        df_pare['macrof1'] = df_pare['macrof1'] * 100

    # Ensure required columns exist
    for df_name, df in [('Score', df_score), ('Pare', df_pare), ('Hiclre', df_hiclre)]:
        for col in ['dataset', 'seed', 'microf1', 'macrof1']:
            if col not in df.columns:
                raise KeyError(f"Missing column '{col}' in {df_name} results")

    # Merge on dataset and seed (inner join keeps only matching pairs)
    merged_sp = df_score.merge(df_pare, on=['dataset', 'seed'], suffixes=('_score', '_pare'), how='inner')
    merged_sh = df_score.merge(df_hiclre, on=['dataset', 'seed'], suffixes=('_score', '_hiclre'), how='inner')

    # Compute differences
    score_vs_pare_df = merged_sp[['dataset', 'seed']].copy()
    score_vs_pare_df['microf1_diff'] = merged_sp['microf1_score'] - merged_sp['microf1_pare']
    score_vs_pare_df['macrof1_diff'] = merged_sp['macrof1_score'] - merged_sp['macrof1_pare']

    score_vs_hiclre_df = merged_sh[['dataset', 'seed']].copy()
    score_vs_hiclre_df['microf1_diff'] = merged_sh['microf1_score'] - merged_sh['microf1_hiclre']
    score_vs_hiclre_df['macrof1_diff'] = merged_sh['macrof1_score'] - merged_sh['macrof1_hiclre']

    return score_vs_pare_df.reset_index(drop=True), score_vs_hiclre_df.reset_index(drop=True)


score_vs_pare, score_vs_hiclre = compute_differences(Score_results, Pare_results, Hiclre_results)
score_vs_pare, score_vs_hiclre


def cliffs_delta(x, y):
    """
    Calcola Cliff's delta tra due array/list.
    Restituisce (delta, interpretazione).
    """
    n_x, n_y = len(x), len(y)
    greater = sum(1 for xi in x for yj in y if xi > yj)
    less    = sum(1 for xi in x for yj in y if xi < yj)
    delta = (greater - less) / (n_x * n_y)

    # interpretazione standard
    size = abs(delta)
    if size < 0.147:
        interp = "trascurabile"
    elif size < 0.33:
        interp = "piccolo"
    elif size < 0.474:
        interp = "medio"
    else:
        interp = "grande"
    return delta, interp

# Confronto Score-Pare

In [13]:
# Create a dataframe with Score_results and Pare_results aligned by dataset and seed
aligned_results = []
for score_entry in Score_results:
    for pare_entry in Pare_results:
        if score_entry['dataset'] == pare_entry['dataset'] and score_entry['seed'] == pare_entry['seed']:
            # Pare results in some cases are in [0,1] — convert to percentages if needed
            pare_micro = pare_entry.get('microf1', np.nan)
            pare_macro = pare_entry.get('macrof1', np.nan)
            if pd.notna(pare_micro) and pare_micro <= 1.0:
                pare_micro = pare_micro * 100.0
            if pd.notna(pare_macro) and pare_macro <= 1.0:
                pare_macro = pare_macro * 100.0

            aligned_results.append({
                'dataset': score_entry['dataset'],
                'seed': score_entry['seed'],
                'score_microf1': float(score_entry.get('microf1', np.nan)),
                'score_macrof1': float(score_entry.get('macrof1', np.nan)),
                'pare_microf1': float(pare_micro),
                'pare_macrof1': float(pare_macro)
            })

# Convert to DataFrame for downstream usage (e.g., wilcoxon expects Series-like inputs)
aligned_results = pd.DataFrame(aligned_results)

# Ensure numeric types
for col in ['score_microf1', 'score_macrof1', 'pare_microf1', 'pare_macrof1']:
    aligned_results[col] = pd.to_numeric(aligned_results[col], errors='coerce')

aligned_results

from scipy.stats import ttest_rel
import numpy as np

dataset = "nyt10d"
method1 = 'score'
method2 = 'pare'

df = aligned_results[aligned_results['dataset'] == dataset].copy()

def compare_methods(df, metric, method1, method2, dataset):
    x = df[f"{method1}_{metric}"].values
    y = df[f"{method2}_{metric}"].values
    n = len(x)

    print(f"\n=== {metric.upper()} ({dataset}, n={n}) ===")

    mean1, mean2 = np.mean(x), np.mean(y)
    diff = mean1 - mean2
    winner = method1 if mean1 > mean2 else method2

    # Cliff's delta (effect size)
    delta, res = cliffs_delta(x, y)
    print(f"Cliff's delta = {delta:.3f} ({res} effect size)")
    print(f"Mean {method1}: {mean1:.4f}, Mean {method2}: {mean2:.4f}, Mean diff: {diff:.4f}")

    # Paired t-test (parametric, più sensibile)
    stat_t, p_t = ttest_rel(x, y)
    print(f"Paired t-test → statistic = {stat_t:.3f}, p-value = {p_t:.4f}")

    # Interpretazione integrata
    if p_t < 0.05:
        print(f"→ Significant difference (t-test p < 0.05): {winner} performs better on {metric}.")
    elif n < 6 and p_t < 0.10:
        print(f"→ Tentative difference (t-test p ≈ {p_t:.4f}, n small={n}): {winner} likely better.")
    else:
        print(f"→ No statistically significant difference (t-test p = {p_t:.4f}).")
        print(f"  However, {winner} shows {res} effect (mean diff = {diff:.4f}).")

# MICRO-F1
compare_methods(df, "microf1", method1, method2, dataset)

# MACRO-F1
compare_methods(df, "macrof1", method1, method2, dataset)


=== MICROF1 (nyt10d, n=5) ===
Cliff's delta = 0.920 (grande effect size)
Mean score: 87.6390, Mean pare: 85.8706, Mean diff: 1.7684
Paired t-test → statistic = 3.035, p-value = 0.0386
→ Significant difference (t-test p < 0.05): score performs better on microf1.

=== MACROF1 (nyt10d, n=5) ===
Cliff's delta = 0.680 (grande effect size)
Mean score: 47.7730, Mean pare: 45.7901, Mean diff: 1.9829
Paired t-test → statistic = 1.312, p-value = 0.2598
→ No statistically significant difference (t-test p = 0.2598).
  However, score shows grande effect (mean diff = 1.9829).


# Confronto Score Hiclre

In [14]:
# Create a dataframe with Score_results and Hiclre_results aligned by dataset and seed
aligned_results = []
for score_entry in Score_results:
    for hiclre_entry in Hiclre_results:
        if score_entry['dataset'] == hiclre_entry['dataset'] and score_entry['seed'] == hiclre_entry['seed']:
            # Hiclre results in some cases are in [0,1] — convert to percentages if needed
            hiclre_micro = hiclre_entry.get('microf1', np.nan)
            hiclre_macro = hiclre_entry.get('macrof1', np.nan)
            if pd.notna(hiclre_micro) and hiclre_micro <= 1.0:
                hiclre_micro = hiclre_micro * 100.0
            if pd.notna(hiclre_macro) and hiclre_macro <= 1.0:
                hiclre_macro = hiclre_macro * 100.0

            aligned_results.append({
                'dataset': score_entry['dataset'],
                'seed': score_entry['seed'],
                'score_microf1': float(score_entry.get('microf1', np.nan)),
                'score_macrof1': float(score_entry.get('macrof1', np.nan)),
                'hiclre_microf1': float(hiclre_micro),
                'hiclre_macrof1': float(hiclre_macro)
            })

# Convert to DataFrame for downstream usage (e.g., wilcoxon expects Series-like inputs)
aligned_results = pd.DataFrame(aligned_results)

# Ensure numeric types
for col in ['score_microf1', 'score_macrof1', 'hiclre_microf1', 'hiclre_macrof1']:
    aligned_results[col] = pd.to_numeric(aligned_results[col], errors='coerce')

aligned_results

from scipy.stats import ttest_rel
import numpy as np

dataset = "nyt10d"
method1 = 'score'
method2 = 'hiclre'

df = aligned_results[aligned_results['dataset'] == dataset].copy()

def compare_methods(df, metric, method1, method2, dataset):
    x = df[f"{method1}_{metric}"].values
    y = df[f"{method2}_{metric}"].values
    n = len(x)

    print(f"\n=== {metric.upper()} ({dataset}, n={n}) ===")

    mean1, mean2 = np.mean(x), np.mean(y)
    diff = mean1 - mean2
    winner = method1 if mean1 > mean2 else method2

    # Cliff's delta (effect size)
    delta, res = cliffs_delta(x, y)
    print(f"Cliff's delta = {delta:.3f} ({res} effect size)")
    print(f"Mean {method1}: {mean1:.4f}, Mean {method2}: {mean2:.4f}, Mean diff: {diff:.4f}")

    # Paired t-test (parametric, più sensibile)
    stat_t, p_t = ttest_rel(x, y)
    print(f"Paired t-test → statistic = {stat_t:.3f}, p-value = {p_t:.4f}")

    # Interpretazione integrata
    if p_t < 0.05:
        print(f"→ Significant difference (t-test p < 0.05): {winner} performs better on {metric}.")
    elif n < 6 and p_t < 0.10:
        print(f"→ Tentative difference (t-test p ≈ {p_t:.4f}, n small={n}): {winner} likely better.")
    else:
        print(f"→ No statistically significant difference (t-test p = {p_t:.4f}).")
        print(f"  However, {winner} shows {res} effect (mean diff = {diff:.4f}).")

# MICRO-F1
compare_methods(df, "microf1", method1, method2, dataset)

# MACRO-F1
compare_methods(df, "macrof1", method1, method2, dataset)


=== MICROF1 (nyt10d, n=5) ===
Cliff's delta = 1.000 (grande effect size)
Mean score: 87.6390, Mean hiclre: 75.7323, Mean diff: 11.9067
Paired t-test → statistic = 23.217, p-value = 0.0000
→ Significant difference (t-test p < 0.05): score performs better on microf1.

=== MACROF1 (nyt10d, n=5) ===
Cliff's delta = 1.000 (grande effect size)
Mean score: 47.7730, Mean hiclre: 20.7141, Mean diff: 27.0589
Paired t-test → statistic = 25.774, p-value = 0.0000
→ Significant difference (t-test p < 0.05): score performs better on macrof1.


# Ceonfronto Pare Hiclre

In [15]:
# Create a dataframe with Score_results and Hiclre_results aligned by dataset and seed
aligned_results = []
for pare_entry in Pare_results:
    for hiclre_entry in Hiclre_results:
        if pare_entry['dataset'] == hiclre_entry['dataset'] and pare_entry['seed'] == hiclre_entry['seed']:
            # Hiclre results in some cases are in [0,1] — convert to percentages if needed
            hiclre_micro = hiclre_entry.get('microf1', np.nan)
            hiclre_macro = hiclre_entry.get('macrof1', np.nan)
            if pd.notna(hiclre_micro) and hiclre_micro <= 1.0:
                hiclre_micro = hiclre_micro * 100.0
            if pd.notna(hiclre_macro) and hiclre_macro <= 1.0:
                hiclre_macro = hiclre_macro * 100.0
            

            aligned_results.append({
                'dataset': pare_entry['dataset'],
                'seed': pare_entry['seed'],
                'pare_microf1': float(pare_entry.get('microf1', np.nan))*100.0,
                'pare_macrof1': float(pare_entry.get('macrof1', np.nan))*100.0,
                'hiclre_microf1': float(hiclre_micro),
                'hiclre_macrof1': float(hiclre_macro)
            })

# Convert to DataFrame for downstream usage (e.g., wilcoxon expects Series-like inputs)
aligned_results = pd.DataFrame(aligned_results)

# Ensure numeric types
for col in ['pare_microf1', 'pare_macrof1', 'hiclre_microf1', 'hiclre_macrof1']:
    aligned_results[col] = pd.to_numeric(aligned_results[col], errors='coerce')

aligned_results

from scipy.stats import ttest_rel
import numpy as np

dataset = "nyt10d"
method1 = 'pare'
method2 = 'hiclre'

df = aligned_results[aligned_results['dataset'] == dataset].copy()

def compare_methods(df, metric, method1, method2, dataset):
    x = df[f"{method1}_{metric}"].values
    y = df[f"{method2}_{metric}"].values
    n = len(x)

    print(f"\n=== {metric.upper()} ({dataset}, n={n}) ===")

    mean1, mean2 = np.mean(x), np.mean(y)
    diff = mean1 - mean2
    winner = method1 if mean1 > mean2 else method2

    # Cliff's delta (effect size)
    delta, res = cliffs_delta(x, y)
    print(f"Cliff's delta = {delta:.3f} ({res} effect size)")
    print(f"Mean {method1}: {mean1:.4f}, Mean {method2}: {mean2:.4f}, Mean diff: {diff:.4f}")

    # Paired t-test (parametric, più sensibile)
    stat_t, p_t = ttest_rel(x, y)
    print(f"Paired t-test → statistic = {stat_t:.3f}, p-value = {p_t:.4f}")

    # Interpretazione integrata
    if p_t < 0.05:
        print(f"→ Significant difference (t-test p < 0.05): {winner} performs better on {metric}.")
    elif n < 6 and p_t < 0.10:
        print(f"→ Tentative difference (t-test p ≈ {p_t:.4f}, n small={n}): {winner} likely better.")
    else:
        print(f"→ No statistically significant difference (t-test p = {p_t:.4f}).")
        print(f"  However, {winner} shows {res} effect (mean diff = {diff:.4f}).")

# MICRO-F1
compare_methods(df, "microf1", method1, method2, dataset)

# MACRO-F1
compare_methods(df, "macrof1", method1, method2, dataset)


=== MICROF1 (nyt10d, n=5) ===
Cliff's delta = 1.000 (grande effect size)
Mean pare: 85.8706, Mean hiclre: 75.7323, Mean diff: 10.1383
Paired t-test → statistic = 19.456, p-value = 0.0000
→ Significant difference (t-test p < 0.05): pare performs better on microf1.

=== MACROF1 (nyt10d, n=5) ===
Cliff's delta = 1.000 (grande effect size)
Mean pare: 45.7901, Mean hiclre: 20.7141, Mean diff: 25.0760
Paired t-test → statistic = 18.247, p-value = 0.0001
→ Significant difference (t-test p < 0.05): pare performs better on macrof1.
